# Binary Search Tree & AVL

*BST · Self-Balancing · 4 Traversals · Real-World*


---
## Balanced Trees — AVL


# Balanced Trees

*Run each cell with **Shift+Enter***

01 — DSA Internals: Balanced Trees (AVL) From Scratch
=====================================================

Runnable companion to PDF Book II "Why a plain BST isn't enough".

A Binary Search Tree gives O(log n) search — but ONLY if it stays balanced.
Insert sorted data into a naive BST and it degenerates into a linked list:
O(n). An AVL tree fixes this by keeping every node's subtree heights within 1,
performing ROTATIONS after each insert to restore balance.

This file builds an AVL tree and PROVES the height stays ~log2(n) even for the
worst-case input (already-sorted keys), where a naive BST would be height n-1.

The four rotation cases:
    LL  -> right rotate
    RR  -> left rotate
    LR  -> left rotate child, then right rotate
    RL  -> right rotate child, then left rotate


---
## 🧠 Mental Model: Binary Search Trees & AVL Trees

> **A BST is a binary tree with a search rule: every left child < parent < every right child.**  
> An AVL tree enforces this rule AND guarantees the tree stays balanced — so the "log n" promise never breaks.

### WHY — Why does it exist?
Arrays: O(1) by index but O(n) to find. Sorted arrays: O(log n) to find but O(n) to insert. A BST gives O(log n) **both** — when it's balanced. Unbalanced BSTs degenerate to O(n) (like a linked list).

### WHAT — What is it?

```
          5              ← root
        /   \
       3     8           ← BST property: all left < 5, all right > 5
      / \   / \
     2   4 7   9         ← height = 3, so ops ≈ log₂(9) ≈ 3 comparisons
```

**AVL Tree** — self-balancing BST that keeps: `|height(left) - height(right)| ≤ 1` at every node. Enforced by 4 rotation types after every insert/delete.

### HOW — The four AVL rotations

```
LL imbalance  →  right rotate
RR imbalance  →  left rotate
LR imbalance  →  left rotate child, then right rotate node
RL imbalance  →  right rotate child, then left rotate node
```

**Traversal order:**
| Order | Visit sequence | Use case |
|-------|---------------|----------|
| In-order (L → N → R) | Sorted ascending | Print sorted values |
| Pre-order (N → L → R) | Root first | Serialize tree |
| Post-order (L → R → N) | Children first | Delete tree, calc size |
| Level-order (BFS) | Breadth-first | Find height, level-K nodes |

### WHEN — When to use BST/AVL?
- **Sorted dynamic set**: need to insert/delete AND query in sorted order (O(log n) all)
- **Range queries**: "find all keys between 10 and 50" — walk in-order
- **Order statistics**: k-th smallest element
- **When to prefer `sortedcontainers.SortedList`**: in Python, this is a B-tree hybrid; almost always better than rolling your own AVL

**Gotcha** — A plain BST with sorted input degenerates to a linked list (height n, O(n) ops). Always use a balanced variant (AVL / Red-Black) or `sortedcontainers.SortedList` in production.

```
Complexity (balanced):
  Search  O(log n)
  Insert  O(log n)
  Delete  O(log n)
  Space   O(n)
```


In [ ]:
from __future__ import annotations

import math


class _Node:
    __slots__ = ("key", "left", "right", "height")

    def __init__(self, key):
        self.key = key
        self.left: _Node | None = None
        self.right: _Node | None = None
        self.height = 1


def _h(n: _Node | None) -> int:
    return n.height if n else 0


def _balance(n: _Node | None) -> int:
    return _h(n.left) - _h(n.right) if n else 0


def _update(n: _Node) -> None:
    n.height = 1 + max(_h(n.left), _h(n.right))


def _rotate_right(y: _Node) -> _Node:
    x = y.left
    assert x is not None
    y.left = x.right
    x.right = y
    _update(y)
    _update(x)
    return x                                # x is the new subtree root


def _rotate_left(x: _Node) -> _Node:
    y = x.right
    assert y is not None
    x.right = y.left
    y.left = x
    _update(x)
    _update(y)
    return y


class AVLTree:
    def __init__(self):
        self._root: _Node | None = None
        self._size = 0

    def insert(self, key) -> None:
        inserted = [False]
        self._root = self._insert(self._root, key, inserted)
        if inserted[0]:
            self._size += 1

    def _insert(self, node: _Node | None, key, inserted) -> _Node:
        if node is None:
            inserted[0] = True
            return _Node(key)
        if key < node.key:
            node.left = self._insert(node.left, key, inserted)
        elif key > node.key:
            node.right = self._insert(node.right, key, inserted)
        else:
            return node                     # duplicate key: ignore

        _update(node)
        bal = _balance(node)

        if bal > 1 and key < node.left.key:            # LL
            return _rotate_right(node)
        if bal < -1 and key > node.right.key:          # RR
            return _rotate_left(node)
        if bal > 1 and key > node.left.key:            # LR
            node.left = _rotate_left(node.left)
            return _rotate_right(node)
        if bal < -1 and key < node.right.key:          # RL
            node.right = _rotate_right(node.right)
            return _rotate_left(node)
        return node

    def __contains__(self, key) -> bool:
        n = self._root
        while n:
            if key == n.key:
                return True
            n = n.left if key < n.key else n.right
        return False

    def height(self) -> int:
        return _h(self._root)

    def inorder(self) -> list:
        out: list = []
        self._walk(self._root, out)
        return out

    def _walk(self, n: _Node | None, out: list) -> None:
        if n:
            self._walk(n.left, out)
            out.append(n.key)
            self._walk(n.right, out)

    def is_balanced(self) -> bool:
        def check(n: _Node | None) -> bool:
            if n is None:
                return True
            return abs(_balance(n)) <= 1 and check(n.left) and check(n.right)
        return check(self._root)

    def __len__(self) -> int:
        return self._size


def demo() -> None:
    # WORST CASE for a naive BST: already-sorted keys would form a chain of
    # height n-1. AVL keeps height near log2(n).
    n = 1000
    tree = AVLTree()
    for k in range(n):
        tree.insert(k)

    assert len(tree) == n
    assert tree.inorder() == list(range(n)), "in-order traversal must be sorted"
    assert tree.is_balanced(), "every node must satisfy the AVL invariant"

    naive_bst_height = n            # sorted insert into plain BST -> ~n
    log_n = math.log2(n)
    assert tree.height() <= 1.5 * log_n, "AVL height must stay near log2(n)"
    print(f"   inserted {n} SORTED keys")
    print(f"   naive BST would be height ~{naive_bst_height};  AVL height = {tree.height()} (log2 n = {log_n:.1f})")

    assert 500 in tree and n not in tree
    print("   search, balance invariant, and sorted traversal all verified")


def main() -> None:
    print("=" * 70)
    print("DSA INTERNALS — balanced_trees.py (AVL)")
    print("=" * 70)
    print("Self-balancing BST via rotations (LL / RR / LR / RL):")
    demo()
    print("-" * 70)
    print("Lesson: a plain BST degrades to O(n) on sorted input; AVL rotations keep it O(log n).")
    print("All balanced_trees demos passed ✔")


if __name__ == "__main__":
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()

---
## Tries — Prefix Trees


# Tries

*Run each cell with **Shift+Enter***

01 — DSA Internals: Trie (Prefix Tree) From Scratch
===================================================

Runnable companion to PDF Book II "The data structure behind autocomplete".

A hash set answers "is this exact word present?" in O(L). A TRIE answers
PREFIX questions — "which words start with 'pre'?" — which a hash set can't do
without scanning everything. Each node is a character; a path from the root
spells a prefix; a flag marks the end of a real word.

Costs (L = length of the key, NOT the number of words):
    insert / search / starts_with : O(L)
    autocomplete                  : O(L + size of the matching subtree)

Used in autocomplete, spell-check, IP routing tables, and T9 text entry.

In [ ]:
from __future__ import annotations


class _TrieNode:
    __slots__ = ("children", "is_word")

    def __init__(self):
        self.children: dict[str, _TrieNode] = {}
        self.is_word = False


class Trie:
    def __init__(self):
        self._root = _TrieNode()
        self._size = 0

    def insert(self, word: str) -> None:
        node = self._root
        for ch in word:
            node = node.children.setdefault(ch, _TrieNode())
        if not node.is_word:
            node.is_word = True
            self._size += 1

    def _find(self, prefix: str) -> _TrieNode | None:
        node = self._root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word: str) -> bool:
        node = self._find(word)
        return node is not None and node.is_word

    def starts_with(self, prefix: str) -> bool:
        return self._find(prefix) is not None

    def autocomplete(self, prefix: str) -> list[str]:
        """All stored words beginning with `prefix`, in sorted order."""
        node = self._find(prefix)
        if node is None:
            return []
        out: list[str] = []
        self._collect(node, prefix, out)
        return sorted(out)

    def _collect(self, node: _TrieNode, path: str, out: list[str]) -> None:
        if node.is_word:
            out.append(path)
        for ch, child in node.children.items():
            self._collect(child, path + ch, out)

    def __len__(self) -> int:
        return self._size

    def __contains__(self, word: str) -> bool:
        return self.search(word)


def demo() -> None:
    words = ["cat", "car", "card", "care", "dog", "dodge", "do", "care"]  # dup 'care'
    trie = Trie()
    for w in words:
        trie.insert(w)

    assert len(trie) == 7, "7 unique words; duplicate 'care' counted once"
    assert trie.search("car") and trie.search("card")
    assert not trie.search("ca"), "'ca' is a prefix, not a stored word"
    assert trie.starts_with("ca") and trie.starts_with("do")
    assert not trie.starts_with("xyz")
    print("   inserted", sorted(set(words)))
    print("   search('car') =", trie.search("car"), " starts_with('ca') =", trie.starts_with("ca"))

    # Autocomplete — the thing a hash set fundamentally cannot do in O(L).
    assert trie.autocomplete("car") == ["car", "card", "care"]
    assert trie.autocomplete("do") == ["do", "dodge", "dog"]
    assert trie.autocomplete("z") == []
    print("   autocomplete('car') =", trie.autocomplete("car"))
    print("   autocomplete('do')  =", trie.autocomplete("do"))


def main() -> None:
    print("=" * 70)
    print("DSA INTERNALS — tries.py (prefix tree)")
    print("=" * 70)
    print("Character-per-node tree; insert/search/starts_with are O(L):")
    demo()
    print("-" * 70)
    print("Lesson: a trie trades memory for O(L) PREFIX queries a hash set can't answer. Autocomplete/routing.")
    print("All tries demos passed ✔")


if __name__ == "__main__":
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()

---
## Exhaustive Gotchas & Real-World Scenarios


BST property: left subtree < node < right subtree.
Operations: O(h) where h = tree height.
  Balanced tree:  h = O(log n)  → fast
  Degenerate tree (sorted input): h = O(n)  → same as linked list!

> ⚠️ **GOTCHA 1: Inserting sorted data into a BST creates a linked list O(n) not O(log n).**
> ⚠️ **GOTCHA 2: BST deletion with two children — replace with in-order SUCCESSOR**
  (minimum of right subtree) to preserve BST property.
> ⚠️ **GOTCHA 3: In-order traversal of a BST gives elements in SORTED order.**
  Use this to verify BST correctness or to extract sorted elements.
> ⚠️ **GOTCHA 4: Python has no built-in BST; use sortedcontainers.SortedList or**
  implement AVL/Red-Black for production.
> ⚠️ **GOTCHA 5: Validating a BST — must pass min/max BOUNDS, not just compare**
  parent and child directly.

In [ ]:
def notebook_bst_gotchas() -> None:

## §4 · BST Gotchas

In [ ]:
# ── §4.1  Degenerate tree on sorted input ─────────────────────────────
    bst_sorted = BST()
    for v in [1, 2, 3, 4, 5]:           # sorted input → O(n) height!
        bst_sorted.insert(v)

    bst_random = BST()
    for v in [3, 1, 5, 2, 4]:           # shuffled → O(log n) height
        bst_random.insert(v)

    h_sorted = bst_sorted.height()
    h_random = bst_random.height()
    print(f"Sorted-insert height:  {h_sorted}  (degenerate — same as linked list!)")
    print(f"Balanced-insert height:{h_random}  (O(log n))")

    # ── §4.2  In-order traversal = sorted output ──────────────────────────
    bst2 = BST()
    for v in [5, 3, 7, 1, 4, 6, 8, 2]: bst2.insert(v)
    inorder = bst2.inorder()
    assert inorder == sorted(inorder)
    print(f"\nIn-order traversal is always SORTED: {inorder}")

    # ── §4.3  GOTCHA: BST validation — need min/max bounds ───────────────
    #
    # Wrong approach: just check node > left child and node < right child.
    # This FAILS for:
    #       5
    #      / \
    #     1   4    <- 4 < 5 BUT 4 < 5 violates the BST property (4 should be
    #        / \      in the right subtree of 5, so must be > 5)
    #       3   6
    #
    # Correct: each node must be in the range (min_bound, max_bound).

    from dataclasses import dataclass as _dc
    @_dc
    class _N:
        key: int
        left: object = None
        right: object = None

    def is_valid_bst(root, lo=float('-inf'), hi=float('inf')):
        if root is None: return True
        v = root.val if hasattr(root, 'val') else root.key
        if not (lo < v < hi): return False
        return (is_valid_bst(root.left, lo, v) and
                is_valid_bst(root.right, v, hi))

    invalid_root = _N(4, _N(6, _N(1)), _N(7))   # 6 > 4 but on left → invalid!
    print(f"'Invalid' tree valid? {is_valid_bst(invalid_root)}")   # False ✓

    # ── §4.4  Why Python has no built-in BST ─────────────────────────────
    print("\nFor sorted ordered operations in Python:")
    print("  sortedcontainers.SortedList — O(log n) add/remove, O(1) index")
    print("  bisect module               — binary search on sorted lists")
    import bisect
    sl = [1, 3, 5, 7, 9]
    bisect.insort(sl, 4)   # insert 4 maintaining sorted order
    print(f"  bisect.insort([1,3,5,7,9], 4) = {sl}")